In [1]:
import numpy as np
from desdeo.tools import PyomoIpoptSolver
from desdeo.tools.scalarization import add_asf_diff
from desdeo.problem.testproblems.dtlz2_problem import dtlz2

from desdeo.gdm.favorite_method import (
    IPR_Options, GPRMOptions, ZoomOptions, FavOptions,
    favorite_method, generate_next_iteration_mps, cluster_points
)
from desdeo.gdm.voting_rules import majority_rule

from visualizations import visualize_3d_clusters


In [2]:
# --- 1. SETUP PROBLEM ---
dtlz2_problem = dtlz2(8, 3)
# Generate random reference points and find MPS
n_of_dms = 3
reference_points = {}
for i in range(n_of_dms):
    reference_points[f"DM{i+1}"] = {"f_1": np.random.random(), "f_2": np.random.random(), "f_3": np.random.random()}

reference_points = {
    "DM1": {"f_1": 0.0, "f_2": 0.9, "f_3": 0.5},
    "DM2": {"f_1": 0.5, "f_2": 0.0, "f_3": 0.9},
    "DM3": {"f_1": 0.9, "f_2": 0.5, "f_3": 0.0},
}
most_preferred_solutions = {}
for dm in reference_points.keys():
    p, target = add_asf_diff(dtlz2_problem, symbol="asf", reference_point=reference_points[dm])
    solver = PyomoIpoptSolver(p)
    res = solver.solve(target)
    most_preferred_solutions[f"{dm}"] = res.optimal_objectives


In [3]:
# --- 2. CONFIGURE INITIAL OPTIONS ---
ipr_options = IPR_Options(
    most_preferred_solutions=most_preferred_solutions,
    num_initial_reference_points=10000,
    version="box",
)
fav_options = FavOptions(
    GPRMoptions=GPRMOptions(method_options=ipr_options),
    candidate_generation_options="mm",
    zoom_options=ZoomOptions(num_steps_remaining=4),
    original_most_preferred_solutions=most_preferred_solutions,
    votes=None,
    total_n_of_candidates=5
)


In [4]:

# --- 3. THE LOOP ---
results_history = []
current_options = fav_options
fractions = [0.8, 0.6, 0.4, 0.2]  # Shrink the hull each iteration

# Run 4 iterations
for iter_idx in range(4):
    print(f"\n{'='*40}\n--- RUNNING ITERATION {iter_idx + 1} ---\n{'='*40}")

    # 1. Run Core Method
    fav_results = favorite_method(
        problem=dtlz2_problem,
        options=current_options,
        results_list=results_history
    )
    results_history.append(fav_results)

    points_matrix, centers_matrix, cluster_labels = cluster_points(fav_results)

    visualize_3d_clusters(
        options=current_options.GPRMoptions, 
        points_arr=points_matrix, 
        centers_arr=centers_matrix, 
        labels=cluster_labels, 
        n_predetermined=len(fav_results.fair_solutions), 
        iter_n=iter_idx+1,
    )
        #current_mps=fav_results.FavOptions.original_most_preferred_solutions

    # 2. DM Voting 
    dm1input = input()
    dm2input = input()
    dm3input = input()
    votes = {"DM1": int(dm1input), "DM2": int(dm2input), "DM3": int(dm3input)}
    winning_idx = majority_rule(votes)

    # 3. Generate Next Iteration MPS and get visualization data!
    next_mps = generate_next_iteration_mps(
        fav_results=fav_results,
        cluster_labels=cluster_labels,
        winning_idx=winning_idx,
        fraction_to_keep=fractions[iter_idx]
    )

    # 5. Prepare Options for Next Iteration
    current_options = current_options.model_copy(deep=True)
    current_options.GPRMoptions.method_options.most_preferred_solutions = next_mps
    current_options.GPRMoptions.method_options.version = "convex_hull"
    current_options.zoom_options.num_steps_remaining = max(1, 4 - iter_idx - 1)
    current_options.votes = votes





--- RUNNING ITERATION 1 ---
Run 1/100
Run 10/100
Run 20/100
Run 30/100
Run 40/100
Run 50/100
Run 60/100
Run 70/100
Run 80/100
Run 90/100
Run 100/100


Expanded set: 80 points selected.

--- RUNNING ITERATION 2 ---
Run 1/100
Run 10/100
Run 20/100
Run 30/100
Run 40/100
Run 50/100
Run 60/100
Run 70/100
Run 80/100
Run 90/100
Run 100/100


Expanded set: 60 points selected.

--- RUNNING ITERATION 3 ---
Run 1/100
Run 10/100
Run 20/100
Run 30/100
Run 40/100
Run 50/100
Run 60/100
Run 70/100
Run 80/100
Run 90/100
Run 100/100


Expanded set: 40 points selected.

--- RUNNING ITERATION 4 ---
Run 1/100
Run 10/100
Run 20/100
Run 30/100
Run 40/100
Run 50/100
Run 60/100
Run 70/100
Run 80/100
Run 90/100
Run 100/100


Expanded set: 20 points selected.


In [ ]:
# TODO: make the final vote set thingy.
# TODO: decide the % of solutions to automatically